# 03 · Congestión y ocupación de urgencias

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src import datos, flow
from src.config import FIGURES, DATA_INTERIM, DATA_PROCESSED, ETAPAS
pd.set_option("display.max_columns", 50); pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid", context="notebook")
def guardar(nombre):
    plt.tight_layout(); plt.savefig(FIGURES / f"{nombre}.png", dpi=150, bbox_inches="tight"); plt.show()


In [ ]:
et = datos.etapas(); ep = datos.episodios(et)
occ = datos.censo_urgencias(et)
occ = occ[(occ.fecha_hora >= "2023-01-08") & (occ.fecha_hora < "2026-05-01")]
occ["hora"] = occ.fecha_hora.dt.hour; occ["dia"] = occ.fecha_hora.dt.dayofweek
occ.ocupacion.describe(percentiles=[.5, .9, .95, .99])

In [ ]:
dias = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
llegadas = ep.assign(hora=ep.hora_llegada, dia=ep.dia_semana).pivot_table(index="hora", columns="dia", values="episodio_id", aggfunc="count")
semanas = ep.inicio.dt.to_period("W").nunique()
sns.heatmap(llegadas / semanas, cmap="YlOrRd", ax=ax[0], xticklabels=dias, annot=False); ax[0].set_title("Llegadas promedio por hora y día")
sns.heatmap(occ.pivot_table(index="hora", columns="dia", values="ocupacion"), cmap="YlOrRd", ax=ax[1], xticklabels=dias); ax[1].set_title("Pacientes presentes en urgencias (promedio)")
guardar("03_heatmaps_llegadas_ocupacion")

In [ ]:
diario = occ.set_index("fecha_hora").ocupacion.resample("D").max()
fig, ax = plt.subplots(figsize=(13, 4))
diario.plot(ax=ax, lw=.6, alpha=.6, label="pico diario"); diario.rolling(28).mean().plot(ax=ax, lw=2, label="media 28 días")
ax.axhline(diario.quantile(.9), color="red", ls="--", label="P90 (umbral saturación)"); ax.legend(); ax.set_title("Pico diario de ocupación en urgencias")
guardar("03_ocupacion_diaria")

### ¿La congestión alarga la estancia?

In [ ]:
urg = ep[(ep.tipo_detalle == "Urgencias con observación") & (ep.los_atipica == 0)].copy()
urg["ocup_q"] = pd.qcut(urg.ocupacion_urg_llegada, 10, duplicates="drop")
efecto = urg.groupby("ocup_q", observed=True).agg(n=("los_h", "size"), estancia_p50=("los_h", "median"), estancia_media=("los_h", "mean"),
    p_reconsulta_72h=("reconsulta_72h", "mean"))
hosp = ep[ep.tipo_episodio == "Urgencias → Hospitalización"].copy()
hosp["ocup_q"] = pd.qcut(hosp.ocupacion_urg_llegada, 10, duplicates="drop")
efecto["espera_hosp_p50"] = hosp.groupby("ocup_q", observed=True).h_hasta_hosp.median().values
efecto

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
x = [f"{i.left:.0f}-{i.right:.0f}" for i in efecto.index]
ax.plot(x, efecto.estancia_p50, marker="o", label="Estancia en urgencias (p50, h)")
ax.plot(x, efecto.espera_hosp_p50, marker="s", label="Espera hasta hospitalización (p50, h)")
ax.set_xlabel("Pacientes en urgencias al llegar (decil)"); ax.legend(); plt.xticks(rotation=45)
guardar("03_congestion_vs_estancia")

In [ ]:
por_franja = ep[ep.los_atipica == 0].groupby(["franja", "tipo_detalle"], observed=True).los_h.median().unstack(); por_franja

In [ ]:
occ.to_parquet(DATA_INTERIM / "ocupacion_horaria.parquet", index=False)